# 06 — SHAP Explainability
## SustAInable | Neighborhood Heat Illness Risk Prediction

---

**Purpose:** Opens the black box. This notebook answers the question that separates a trustworthy civic tool from an opaque algorithm:

> *Why did the model flag this ZIP code?*

SHAP (SHapley Additive exPlanations) assigns each feature a score that represents how much it pushed the model's prediction up or down for a specific ZCTA. Unlike feature importance (which tells you what the model used globally), SHAP tells you what drove each individual prediction.

This is what allows a community organizer in West Philadelphia to say: *"The model flagged 19139 because of high diabetes prevalence and aging housing stock — not because of some number we can't trace."*

**Inputs:**
- `outputs/sustainable_model.json` ← trained model from Notebook 04
- `outputs/sustainable_model_metadata.json` ← config from Notebook 04
- `data/processed/sustainable_labeled.csv` ← labeled dataset from Notebook 03

**Outputs:**
- `outputs/06a_shap_summary.png` — global feature importance via SHAP
- `outputs/06b_shap_beeswarm.png` — how each feature affects the score direction
- `outputs/06c_shap_dependence_top3.png` — how the top 3 features interact with score
- `outputs/06d_shap_waterfall_examples.png` — individual ZCTA explanations
- `outputs/06e_equity_audit.png` — does the model flag the right neighborhoods?
- `outputs/sustainable_shap_values.csv` — full SHAP values for every ZCTA-event row

---

> ⚠️ **Prerequisites:** Notebooks 02–05 must have been run first,
> and packages must be installed:
> ```
> pip install xgboost scikit-learn imbalanced-learn shap
> ```
> Note the addition of `shap` — this notebook requires it specifically.

**Author:** Nico Meyering, MPA | Equitech Futures Civic Tech Institute, CTI 2026

---

## Step 0: Imports, Dependency Check, and Preflight

`shap` is the one new package this notebook adds. Everything else carries over from Notebooks 04 and 05.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

READY = True

# ── Dependency check ───────────────────────────────────────────────────────────
print("Checking dependencies...")
missing_pkgs = []

try:
    import xgboost as xgb
    print(f"  ✅ xgboost          {xgb.__version__}")
except ImportError:
    missing_pkgs.append("xgboost")
    print("  ❌ xgboost          — not installed")
    READY = False

try:
    import sklearn
    from sklearn.metrics import roc_auc_score, recall_score
    print(f"  ✅ scikit-learn     {sklearn.__version__}")
except ImportError:
    missing_pkgs.append("scikit-learn")
    print("  ❌ scikit-learn     — not installed")
    READY = False

try:
    import shap
    print(f"  ✅ shap             {shap.__version__}")
    SHAP_AVAILABLE = True
except ImportError:
    missing_pkgs.append("shap")
    print("  ❌ shap             — not installed  ← NEW requirement for this notebook")
    READY = False
    SHAP_AVAILABLE = False

if missing_pkgs:
    print()
    print("🔴 Install missing packages, then restart your kernel and re-run:")
    print(f"   pip install {' '.join(missing_pkgs)}")
else:
    print()
    print("✅ All dependencies satisfied.")

# ── File preflight ─────────────────────────────────────────────────────────────
DATA_PROCESSED = Path("../data/processed")
OUTPUTS        = Path("../outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)

LABELED_PATH  = DATA_PROCESSED / "sustainable_labeled.csv"
MODEL_PATH    = OUTPUTS / "sustainable_model.json"
METADATA_PATH = OUTPUTS / "sustainable_model_metadata.json"

print()
print("Checking required files...")
for label, path in [
    ("Labeled dataset (Notebook 03)", LABELED_PATH),
    ("Trained model  (Notebook 04)",  MODEL_PATH),
    ("Model metadata (Notebook 04)",  METADATA_PATH),
]:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  ✅ {label:<35} ({size_kb:.1f} KB)")
    else:
        print(f"  🔴 {label:<35} NOT FOUND — run Notebook 04 first")
        READY = False

if not READY:
    print()
    print("🔴 PREFLIGHT FAILED.")
    print("   Most likely fix: run Notebook 04, then re-open this notebook.")
else:
    print()
    print("✅ Preflight passed. Ready to explain.")

## Step 1: Load Model, Metadata, and Data

In [ ]:
model = metadata = labeled_df = None
FEATURE_NAMES = []
FINAL_THRESHOLD = 0.30
IS_PROXY = False

if READY:
    # Model
    model = xgb.XGBClassifier()
    model.load_model(str(MODEL_PATH))

    # Metadata
    with open(METADATA_PATH) as f:
        metadata = json.load(f)
    FEATURE_NAMES   = metadata.get("feature_names", [])
    FINAL_THRESHOLD = metadata.get("decision_threshold", 0.30)
    IS_PROXY        = metadata.get("label_mode", "real") == "proxy"

    # Labeled data
    labeled_df = pd.read_csv(LABELED_PATH, dtype={'ZCTA': str})
    labeled_df['ZCTA'] = labeled_df['ZCTA'].str.zfill(5)

    # Add any missing features as 0
    for f in FEATURE_NAMES:
        if f not in labeled_df.columns:
            labeled_df[f] = 0

    print(f"✅ Model loaded       ({MODEL_PATH.name})")
    print(f"✅ Metadata loaded    ({len(FEATURE_NAMES)} features, threshold={FINAL_THRESHOLD})")
    print(f"✅ Labeled data loaded ({len(labeled_df):,} rows)")
    print(f"   Label mode: {'⚠️  PROXY' if IS_PROXY else '✅  REAL (NSSP)'}")

    # Split
    train_df   = labeled_df[labeled_df['split'] == 'train'].copy()
    holdout_df = labeled_df[labeled_df['split'] == 'holdout'].copy()

    X_train   = train_df[FEATURE_NAMES].fillna(0)
    X_holdout = holdout_df[FEATURE_NAMES].fillna(0)
    Y_holdout = holdout_df['label']

    # Score holdout
    holdout_probs = model.predict_proba(X_holdout)[:, 1]
    holdout_preds = (holdout_probs >= FINAL_THRESHOLD).astype(int)
    holdout_df    = holdout_df.copy()
    holdout_df['prob_score'] = holdout_probs
    holdout_df['prediction'] = holdout_preds

    print()
    print(f"Holdout set scored:   {len(holdout_df):,} ZCTA-event rows")
    h_auc = roc_auc_score(Y_holdout, holdout_probs)
    h_rec = recall_score(Y_holdout, holdout_preds, zero_division=0)
    print(f"AUC-ROC: {h_auc:.4f}  |  Recall: {h_rec:.4f}  (consistent with Notebook 05 ✅)")
else:
    print("⚠️  Load skipped — preflight failed.")

## Step 2: Build the SHAP Explainer

SHAP uses a game-theory concept called Shapley values. The analogy: imagine each feature is a player on a team, and the team's "score" is the model's prediction. SHAP figures out how much each player *contributed* to that score by trying every possible combination of teammates. The contribution that holds up across all combinations is the Shapley value.

We use `TreeExplainer` — a fast version designed specifically for tree-based models like XGBoost. It computes exact SHAP values in seconds rather than the hours a brute-force approach would take.

> **Note:** This cell computes SHAP values for the entire holdout set. Depending on your machine and the number of features, this takes 30 seconds to 3 minutes. It runs once — values are cached for all subsequent steps.

In [ ]:
explainer   = None
shap_values = None
shap_df     = None

if READY and SHAP_AVAILABLE and model is not None:
    print("Building TreeExplainer...")
    # Use training data as background for SHAP (standard practice)
    # Sample up to 500 rows to keep background computation fast
    bg_sample = X_train.sample(min(500, len(X_train)), random_state=42)
    explainer = shap.TreeExplainer(model, bg_sample)

    print(f"Computing SHAP values for {len(X_holdout):,} holdout rows...")
    print("(This may take 30 seconds to 3 minutes — grab a coffee ☕)")
    shap_output = explainer(X_holdout)

    # Handle both old-style array and new-style Explanation object
    if hasattr(shap_output, 'values'):
        shap_values = shap_output.values
        base_values = shap_output.base_values
    else:
        shap_values = shap_output
        base_values = np.full(len(X_holdout), explainer.expected_value)

    # For binary classification XGBoost may return shape (n, features, 2)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    print(f"✅ SHAP values computed.")
    print(f"   Shape: {shap_values.shape}  (rows × features)")
    print(f"   Base value (expected model output): {float(np.mean(base_values)):.4f}")

    # Build a tidy dataframe of SHAP values
    shap_df = pd.DataFrame(shap_values, columns=FEATURE_NAMES)
    shap_df.insert(0, 'ZCTA', holdout_df['ZCTA'].values)
    shap_df.insert(1, 'event_id', holdout_df['event_id'].values if 'event_id' in holdout_df.columns else 'unknown')
    shap_df.insert(2, 'label', holdout_df['label'].values)
    shap_df.insert(3, 'prob_score', holdout_df['prob_score'].values)

    print()
    print("SHAP value sample (first 3 rows, first 5 features):")
    preview_cols = ['ZCTA', 'label', 'prob_score'] + FEATURE_NAMES[:5]
    display(shap_df[preview_cols].head(3))

elif not SHAP_AVAILABLE:
    print("⚠️  SHAP not available. Install with: pip install shap")
else:
    print("⚠️  SHAP computation skipped — model or data not loaded.")

## Step 3: Global Feature Importance — SHAP Summary

This chart shows the mean absolute SHAP value for each feature across all holdout observations — a global view of what the model relies on most.

**How to read it:** Features at the top matter most to the model's predictions on average. This is more trustworthy than the gain-based importance from Notebook 04 because it accounts for interaction effects between features.

In [ ]:
if shap_values is not None:
    # Mean absolute SHAP per feature
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature':        FEATURE_NAMES,
        'mean_abs_shap':  mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    importance_df['rank'] = range(1, len(importance_df) + 1)
    importance_df['pct_of_total'] = (
        importance_df['mean_abs_shap'] /
        importance_df['mean_abs_shap'].sum() * 100
    ).round(2)

    top_n = min(20, len(importance_df))
    top_df = importance_df.head(top_n)

    print(f"Top {top_n} Features by Mean |SHAP| Value:")
    display(top_df[['rank','feature','mean_abs_shap','pct_of_total']].reset_index(drop=True))

    # ── Bar chart ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, max(6, top_n * 0.45)))

    palette = ['#e74c3c' if i < 3 else '#3498db' if i < 10 else '#95a5a6'
               for i in range(top_n)]
    bars = ax.barh(
        top_df['feature'][::-1],
        top_df['mean_abs_shap'][::-1],
        color=palette[::-1], edgecolor='white', alpha=0.88
    )
    for bar, val in zip(bars, top_df['mean_abs_shap'][::-1]):
        ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)

    ax.set_xlabel('Mean |SHAP Value|  (average impact on model output)')
    ax.set_title(
        f'SustAInable — Global Feature Importance (SHAP)
'
        f'Top {top_n} of {len(FEATURE_NAMES)} features  |  red = top 3',
        fontsize=12
    )
    # Cumulative coverage annotation
    top3_pct = top_df.head(3)['pct_of_total'].sum()
    top10_pct = top_df.head(min(10, top_n))['pct_of_total'].sum()
    ax.text(0.97, 0.03,
            f'Top 3:  {top3_pct:.1f}% of total SHAP\nTop 10: {top10_pct:.1f}% of total SHAP',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(OUTPUTS / '06a_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/06a_shap_summary.png")
else:
    print("⚠️  SHAP summary skipped — values not computed.")

## Step 4: SHAP Beeswarm — Direction and Magnitude

The summary bar chart told us *which* features matter most. The beeswarm tells us *how* they affect predictions — which direction, and how consistently.

**How to read it:**
- Each dot is one ZCTA-event observation
- **Right of center** (positive SHAP) → feature pushed the prediction *toward elevated*
- **Left of center** (negative SHAP) → feature pushed the prediction *away from elevated*
- **Dot color** → feature value (red = high, blue = low)

**What we want to see:** High values of vulnerability features (diabetes, poverty, aging housing) pushing predictions right. That means the model is working as designed.

In [ ]:
if shap_values is not None:
    top_n_beeswarm = min(15, len(FEATURE_NAMES))

    # Get top features by mean abs SHAP
    mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:top_n_beeswarm]
    top_feat_names = [FEATURE_NAMES[i] for i in top_idx]
    top_shap_vals  = shap_values[:, top_idx]
    top_feat_vals  = X_holdout.values[:, top_idx]

    fig, ax = plt.subplots(figsize=(12, max(6, top_n_beeswarm * 0.55)))

    # Draw each feature row as a beeswarm strip
    for row_idx, feat_idx in enumerate(range(top_n_beeswarm - 1, -1, -1)):
        sv   = top_shap_vals[:, feat_idx]
        fv   = top_feat_vals[:, feat_idx]

        # Normalize feature values 0→1 for color mapping
        fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)

        # Add jitter so points don't overlap
        jitter = np.random.uniform(-0.25, 0.25, size=len(sv))

        scatter = ax.scatter(
            sv,
            row_idx + jitter,
            c=fv_norm, cmap='coolwarm', alpha=0.5, s=8,
            vmin=0, vmax=1
        )

    ax.set_yticks(range(top_n_beeswarm))
    ax.set_yticklabels(top_feat_names[::-1], fontsize=9)
    ax.axvline(0, color='#2c3e50', linewidth=1.5, linestyle='--', alpha=0.7)
    ax.set_xlabel('SHAP Value  (positive = pushes toward elevated illness)',
                  fontsize=10)
    ax.set_title(
        f'SustAInable — SHAP Beeswarm Plot
'
        f'Top {top_n_beeswarm} features  |  Red dots = high feature value, Blue = low',
        fontsize=12
    )

    cbar = plt.colorbar(scatter, ax=ax, pad=0.02, fraction=0.02)
    cbar.set_label('Feature Value
(relative, 0=low, 1=high)', fontsize=8)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Low', 'Mid', 'High'])

    # Annotations
    ax.text(0.98, 0.98, '← Lowers risk score', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#3498db', style='italic')
    ax.text(0.02, 0.98, 'Raises risk score →', transform=ax.transAxes,
            ha='left', va='top', fontsize=8, color='#e74c3c', style='italic')

    plt.tight_layout()
    plt.savefig(OUTPUTS / '06b_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/06b_shap_beeswarm.png")
else:
    print("⚠️  Beeswarm skipped — SHAP values not computed.")

## Step 5: SHAP Dependence Plots — Top 3 Features

Dependence plots show the relationship between a feature's raw value and its SHAP impact — and reveal whether the relationship is linear, threshold-based, or more complex.

**Why this matters for equity work:** If the diabetes SHAP curve suddenly jumps at a certain prevalence level, that's the threshold above which a neighborhood tips from "manageable risk" to "elevated risk." That threshold is something a community health worker can act on.

In [ ]:
if shap_values is not None and len(FEATURE_NAMES) >= 1:
    mean_abs    = np.abs(shap_values).mean(axis=0)
    top3_idx    = np.argsort(mean_abs)[::-1][:3]
    top3_names  = [FEATURE_NAMES[i] for i in top3_idx]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        'SustAInable — SHAP Dependence Plots (Top 3 Features)',
        fontsize=13, fontweight='bold'
    )

    for plot_idx, feat_idx in enumerate(top3_idx):
        feat_name = FEATURE_NAMES[feat_idx]
        feat_vals = X_holdout.iloc[:, feat_idx].values
        sv        = shap_values[:, feat_idx]

        ax = axes[plot_idx]

        # Color by label to show separation
        colors = ['#e74c3c' if l == 1 else '#3498db'
                  for l in holdout_df['label'].values]
        ax.scatter(feat_vals, sv, c=colors, alpha=0.4, s=12)

        # Trend line
        try:
            z = np.polyfit(feat_vals, sv, 2)
            p = np.poly1d(z)
            x_line = np.linspace(feat_vals.min(), feat_vals.max(), 100)
            ax.plot(x_line, p(x_line), color='#2c3e50', linewidth=2,
                    linestyle='--', label='Trend', alpha=0.8)
        except Exception:
            pass

        ax.axhline(0, color='#95a5a6', linewidth=1, linestyle=':')

        # Clean up feature name for display
        display_name = (feat_name
                        .replace('places_', '')
                        .replace('hhi_', 'HHI: ')
                        .replace('acs_', 'ACS: ')
                        .replace('_', ' ')
                        .title())
        ax.set_xlabel(f'{display_name}
(raw feature value)', fontsize=9)
        ax.set_ylabel('SHAP Value', fontsize=9)
        ax.set_title(f'#{plot_idx+1}: {display_name}', fontsize=10)

        red_patch  = mpatches.Patch(color='#e74c3c', alpha=0.6, label='Elevated (label=1)')
        blue_patch = mpatches.Patch(color='#3498db', alpha=0.6, label='Not elevated (label=0)')
        ax.legend(handles=[red_patch, blue_patch], fontsize=7, loc='upper left')

    plt.tight_layout()
    plt.savefig(OUTPUTS / '06c_shap_dependence_top3.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/06c_shap_dependence_top3.png")
    print()
    print("Top 3 features driving predictions:")
    for i, name in enumerate(top3_names):
        mean_impact = mean_abs[top3_idx[i]]
        print(f"  #{i+1}: {name}  (mean |SHAP| = {mean_impact:.4f})")
else:
    print("⚠️  Dependence plots skipped — SHAP values not computed.")

## Step 6: Individual ZCTA Explanations — Waterfall Charts

This is the most operationally valuable output in the entire notebook. A waterfall chart shows exactly why the model gave *one specific ZIP code* its score — feature by feature.

We show four examples:
1. **High-confidence true positive** — model correctly flagged an elevated ZCTA,    high score
2. **Low-confidence true positive** — model correctly flagged, but score was near    the threshold
3. **False negative** — a ZCTA that experienced elevated illness the model missed
4. **Top-scoring ZCTA overall** — whatever ZIP code got the highest score

This is what you show a public health partner when they ask *"why did your model flag our neighborhood?"*

In [ ]:
if shap_values is not None and shap_df is not None:
    hd = holdout_df.copy()
    hd['shap_row_idx'] = range(len(hd))

    # ── Find the four example ZCTAs ────────────────────────────────────────────
    true_pos  = hd[(hd['label'] == 1) & (hd['prediction'] == 1)]
    false_neg = hd[(hd['label'] == 1) & (hd['prediction'] == 0)]

    examples = {}

    # 1. High-confidence TP (highest score among correct positives)
    if len(true_pos) > 0:
        idx = true_pos['prob_score'].idxmax()
        examples['High-Confidence True Positive'] = (
            int(true_pos.loc[idx, 'shap_row_idx']),
            true_pos.loc[idx, 'ZCTA'],
            float(true_pos.loc[idx, 'prob_score'])
        )

    # 2. Low-confidence TP (score closest to threshold among correct positives)
    if len(true_pos) > 1:
        scores = true_pos['prob_score']
        near_thresh = scores[
            (scores >= FINAL_THRESHOLD) &
            (scores < FINAL_THRESHOLD + 0.15)
        ]
        if len(near_thresh) > 0:
            idx = near_thresh.index[
                np.argmin(np.abs(near_thresh.values - FINAL_THRESHOLD))
            ]
        else:
            idx = scores.idxmin()
        examples['Near-Threshold True Positive'] = (
            int(true_pos.loc[idx, 'shap_row_idx']),
            true_pos.loc[idx, 'ZCTA'],
            float(true_pos.loc[idx, 'prob_score'])
        )

    # 3. False negative (highest score among missed positives)
    if len(false_neg) > 0:
        idx = false_neg['prob_score'].idxmax()
        examples['False Negative (Missed Elevation)'] = (
            int(false_neg.loc[idx, 'shap_row_idx']),
            false_neg.loc[idx, 'ZCTA'],
            float(false_neg.loc[idx, 'prob_score'])
        )

    # 4. Top-scoring ZCTA overall
    idx = hd['prob_score'].idxmax()
    examples['Highest-Scoring ZCTA Overall'] = (
        int(hd.loc[idx, 'shap_row_idx']),
        hd.loc[idx, 'ZCTA'],
        float(hd.loc[idx, 'prob_score'])
    )

    # ── Build waterfall for each example ──────────────────────────────────────
    n_examples = len(examples)
    fig, axes  = plt.subplots(1, n_examples, figsize=(n_examples * 6, 9))
    if n_examples == 1:
        axes = [axes]
    fig.suptitle(
        'SustAInable — Individual ZCTA Explanations (Waterfall)',
        fontsize=13, fontweight='bold'
    )

    for ax, (title, (row_idx, zcta, score)) in zip(axes, examples.items()):
        sv_row   = shap_values[row_idx]
        fv_row   = X_holdout.iloc[row_idx].values

        # Top 8 features by absolute SHAP for this row
        top_k    = min(8, len(sv_row))
        top_feat = np.argsort(np.abs(sv_row))[::-1][:top_k]

        sv_top   = sv_row[top_feat]
        fn_top   = [FEATURE_NAMES[i] for i in top_feat]
        fv_top   = fv_row[top_feat]
        other_sv = sv_row[np.argsort(np.abs(sv_row))[::-1][top_k:]]
        other_sum= other_sv.sum()

        # Build waterfall values
        feat_labels = [f.replace('places_','').replace('hhi_','HHI:')
                        .replace('acs_','ACS:').replace('_',' ')[:22]
                       for f in fn_top]
        feat_labels_full = [
            f"{fl}
({fv:.2f})" for fl, fv in zip(feat_labels, fv_top)
        ]
        if abs(other_sum) > 0.001:
            sv_plot  = np.append(sv_top, other_sum)
            fl_plot  = feat_labels_full + [f'Other ({len(other_sv)} features)']
        else:
            sv_plot = sv_top
            fl_plot = feat_labels_full

        bar_colors = ['#e74c3c' if v > 0 else '#3498db' for v in sv_plot]
        y_pos = range(len(sv_plot) - 1, -1, -1)

        bars = ax.barh(list(y_pos), sv_plot,
                       color=bar_colors, edgecolor='white', alpha=0.85)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(fl_plot[::-1], fontsize=7)
        ax.axvline(0, color='#2c3e50', linewidth=1.5)

        for bar, val in zip(bars, sv_plot):
            x_pos_label = val + 0.002 if val >= 0 else val - 0.002
            ha = 'left' if val >= 0 else 'right'
            ax.text(x_pos_label, bar.get_y() + bar.get_height()/2,
                    f'{val:+.3f}', va='center', fontsize=7, ha=ha)

        label_val = int(holdout_df['label'].iloc[row_idx])
        label_str = "Elevated ✅" if label_val == 1 else "Not Elevated"
        color_map = {
            'High-Confidence True Positive':  '#2ecc71',
            'Near-Threshold True Positive':   '#f39c12',
            'False Negative (Missed Elevation)': '#e74c3c',
            'Highest-Scoring ZCTA Overall':   '#9b59b6',
        }
        title_color = color_map.get(title, '#2c3e50')
        ax.set_title(
            f'{title}
ZCTA: {zcta}  |  Score: {score:.3f}  |  Actual: {label_str}',
            fontsize=8, fontweight='bold', color=title_color
        )
        ax.set_xlabel('SHAP Value', fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '06d_shap_waterfall_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/06d_shap_waterfall_examples.png")
    print()
    print("ZCTAs explained:")
    for title, (_, zcta, score) in examples.items():
        print(f"  · {title}: ZCTA {zcta}  (score: {score:.3f})")
else:
    print("⚠️  Waterfall skipped — SHAP values not computed.")

## Step 7: Equity Audit

This is the most important step in the notebook for SustAInable's mission.

An equity audit asks: *is the model flagging the right neighborhoods?* Specifically — are high-vulnerability ZCTAs (high poverty, high disability prevalence, high chronic disease burden) getting higher risk scores? Or is the model inadvertently deprioritizing exactly the communities it was designed to serve?

> If the model were performing well on aggregate metrics but consistently underscoring high-poverty ZCTAs, that would be a serious equity failure — even if the AUC-ROC looks fine on paper. This audit catches that.

In [ ]:
if shap_values is not None and holdout_df is not None:
    audit_df = holdout_df[['ZCTA', 'prob_score', 'label', 'prediction']].copy()

    # ── Identify equity-relevant features ─────────────────────────────────────
    equity_keywords = {
        'poverty':    ['poverty', 'pov', 'income'],
        'disability': ['disability', 'disabled', 'disab'],
        'chronic':    ['diabetes', 'copd', 'chd', 'obesity', 'bphigh', 'casthma'],
        'elderly':    ['elderly', 'age65', 'senior', 'older'],
        'housing':    ['housing', 'renter', 'units', 'age_housing', 'built'],
        'insurance':  ['access2', 'insurance', 'uninsured'],
    }

    # Build composite equity indicators from available features
    for label, kws in equity_keywords.items():
        matching = [
            c for c in FEATURE_NAMES
            if any(kw in c.lower() for kw in kws)
            and not c.endswith('_missing')
        ]
        if matching:
            vals = X_holdout[matching].fillna(0).mean(axis=1).values
            # Normalize 0-1
            vmin, vmax = vals.min(), vals.max()
            if vmax > vmin:
                audit_df[f'equity_{label}'] = (vals - vmin) / (vmax - vmin)
            else:
                audit_df[f'equity_{label}'] = 0.5

    equity_cols = [c for c in audit_df.columns if c.startswith('equity_')]

    if not equity_cols:
        print("ℹ️  No equity-relevant features found in the feature matrix.")
        print("   This audit will run once ACS/HHI data is merged in Notebook 02.")
    else:
        print(f"Equity dimensions identified: {len(equity_cols)}")
        for c in equity_cols:
            print(f"  · {c.replace('equity_', '')}")

        fig, axes = plt.subplots(
            1, min(len(equity_cols), 3),
            figsize=(6 * min(len(equity_cols), 3), 5)
        )
        if len(equity_cols) == 1:
            axes = [axes]
        fig.suptitle(
            'SustAInable — Equity Audit
'
            'Do high-vulnerability ZCTAs get higher risk scores?',
            fontsize=13, fontweight='bold'
        )

        for ax, col in zip(axes, equity_cols[:3]):
            dim_name = col.replace('equity_', '').title()

            # Bin into quartiles
            audit_df['vuln_quartile'] = pd.qcut(
                audit_df[col], q=4,
                labels=['Q1
(Low)', 'Q2', 'Q3', 'Q4
(High)'],
                duplicates='drop'
            )

            quartile_summary = audit_df.groupby(
                'vuln_quartile', observed=True
            )['prob_score'].agg(['mean', 'median', 'std']).reset_index()

            colors_q = ['#3498db', '#f39c12', '#e67e22', '#e74c3c']
            bars = ax.bar(
                range(len(quartile_summary)),
                quartile_summary['mean'],
                color=colors_q[:len(quartile_summary)],
                edgecolor='white',
                alpha=0.85,
                yerr=quartile_summary['std'],
                capsize=5
            )
            ax.set_xticks(range(len(quartile_summary)))
            ax.set_xticklabels(quartile_summary['vuln_quartile'], fontsize=9)
            ax.set_ylabel('Mean SustAInable Risk Score')
            ax.set_title(f'{dim_name} Vulnerability
vs. Risk Score', fontsize=10)
            ax.set_ylim(0, min(1.0, audit_df['prob_score'].max() * 1.3))

            for bar, val in zip(bars, quartile_summary['mean']):
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{val:.3f}',
                    ha='center', fontsize=9, fontweight='bold'
                )

            # Check if trend is correct direction (higher vuln = higher score)
            if len(quartile_summary) >= 2:
                q1_mean = quartile_summary['mean'].iloc[0]
                q4_mean = quartile_summary['mean'].iloc[-1]
                if q4_mean > q1_mean:
                    ax.text(0.97, 0.05,
                            '✅ Correct direction
(high vuln → high score)',
                            transform=ax.transAxes, ha='right', va='bottom',
                            fontsize=8, color='#27ae60',
                            bbox=dict(boxstyle='round,pad=0.3',
                                      facecolor='white', alpha=0.8))
                else:
                    ax.text(0.97, 0.05,
                            '⚠️  Inverse direction
Review feature weights',
                            transform=ax.transAxes, ha='right', va='bottom',
                            fontsize=8, color='#e74c3c',
                            bbox=dict(boxstyle='round,pad=0.3',
                                      facecolor='white', alpha=0.8))

        plt.tight_layout()
        plt.savefig(OUTPUTS / '06e_equity_audit.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("✅ Chart saved to outputs/06e_equity_audit.png")
        print()
        print("Equity Audit Summary:")
        for col in equity_cols:
            dim = col.replace('equity_', '')
            audit_df['vq'] = pd.qcut(
                audit_df[col], q=4, labels=[1,2,3,4], duplicates='drop'
            )
            q_means = audit_df.groupby('vq', observed=True)['prob_score'].mean()
            if len(q_means) >= 2:
                direction = "✅ Correct" if q_means.iloc[-1] > q_means.iloc[0] else "⚠️  Inverse"
                diff = q_means.iloc[-1] - q_means.iloc[0]
                print(f"  {dim:<12}  Q1 mean: {q_means.iloc[0]:.3f}  →  "
                      f"Q4 mean: {q_means.iloc[-1]:.3f}  "
                      f"(Δ={diff:+.3f})  {direction}")
else:
    print("⚠️  Equity audit skipped — SHAP values not computed.")

## Step 8: Save SHAP Values

Exports the full SHAP value matrix to CSV. This enables:
- Building a deployment-ready scoring pipeline that returns explainability   alongside every prediction
- Per-ZCTA explanation reports for partner organizations
- Future analysis without recomputing SHAP from scratch

In [ ]:
if shap_df is not None:
    shap_output_path = OUTPUTS / "sustainable_shap_values.csv"
    shap_df.to_csv(shap_output_path, index=False)

    size_mb = shap_output_path.stat().st_size / (1024 * 1024)
    print(f"✅ SHAP values saved: {shap_output_path}")
    print(f"   Rows:    {len(shap_df):,}  (one per ZCTA-event observation)")
    print(f"   Columns: {shap_df.shape[1]}  (ZCTA, event_id, label, score + one per feature)")
    print(f"   Size:    {size_mb:.2f} MB")
    print()
    print("How to use this file:")
    print("  · Load it to explain any individual prediction without re-running this notebook")
    print("  · Filter to a specific ZCTA to build a per-neighborhood explanation report")
    print("  · Feed into a dashboard for community partner use")
    print()

    # ── Check outputs folder completeness ─────────────────────────────────────
    print("Full outputs inventory:")
    expected = [
        '04a_cv_results.png',
        '04b_threshold_curve.png',
        '04c_feature_importance.png',
        '05a_roc_curve.png',
        '05b_confusion_matrix.png',
        '05c_score_distribution.png',
        '05d_per_event_performance.png',
        '05e_calibration_curve.png',
        '06a_shap_summary.png',
        '06b_shap_beeswarm.png',
        '06c_shap_dependence_top3.png',
        '06d_shap_waterfall_examples.png',
        '06e_equity_audit.png',
        'sustainable_model.json',
        'sustainable_model_metadata.json',
        'sustainable_shap_values.csv',
    ]
    for fname in expected:
        path  = OUTPUTS / fname
        check = "✅" if path.exists() else "⬜ not yet generated"
        print(f"  {check}  {fname}")
else:
    print("⚠️  Nothing to save — SHAP values not computed.")

---

## ✅ What You Just Built

The complete SustAInable notebook pipeline. Six notebooks. One end-to-end supervised ML system that takes public federal data and outputs a ranked, explainable, equity-audited probability score for every ZIP code in the United States — deployable 48–72 hours before a heat event peaks.

---

## The Full Pipeline

| Notebook | What It Does | Key Output |
|---|---|---|
| `02_merge_features` | Merges CDC PLACES, HHI, ACS into feature matrix | `sustainable_features_merged.csv` |
| `03_label_construction` | Builds the Y variable from NSSP or proxy | `sustainable_labeled.csv` |
| `04_model_training` | Trains XGBoost with SMOTE + class weighting | `sustainable_model.json` |
| `05_model_evaluation` | ROC, confusion matrix, calibration, per-event | 5 publication charts |
| `06_shap_explainability` | SHAP global + individual + equity audit | `sustainable_shap_values.csv` + 5 charts |

---

## 🔜 What Comes Next for SustAInable

Three things that would meaningfully advance this project beyond the pipeline:

1. **NSSP data access** — replace proxy labels with real ED visit data.    Every metric in notebooks 05 and 06 will sharpen.    Start at: https://www.cdc.gov/nssp/php/onboarding/index.html

2. **A pilot partner** — one city's public health or emergency management    department running SustAInable against their own data and their own    heat event history. Philadelphia ODP, NYC DOHMH, or Houston Health    Department are natural candidates given event history and data infrastructure.

3. **A deployment interface** — even a simple Streamlit app or API endpoint    that takes a NOAA HeatRisk Level 3+ alert as input and returns a    scored, ranked ZCTA list as output. That's the product.    Everything built here is the engine.

---

*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*  
*Nico Meyering, MPA*